In [0]:
# %pip install -U --quiet databricks-sdk==0.49.0


In [0]:
# %pip install -U --quiet "databricks-langchain>=0.4.0"


In [0]:
# %pip install -U --quiet databricks-agents


In [0]:
# %pip install -U --quiet "mlflow[databricks]>=3.10.1"


In [0]:
# %pip install -U --quiet databricks-vectorsearch


In [0]:
# %pip install -U --quiet langchain==0.3.25
# %pip install -U --quiet langchain-core>=0.3.59

In [0]:
# %pip install -U --quiet bs4==0.0.2


In [0]:
# %pip install -U --quiet markdownify==0.14.1


In [0]:
# %pip install -U --quiet pydantic==2.10.1


In [0]:
# %pip install -U --quiet openai


In [0]:
# %pip install -U --quiet PyMuPDF

In [0]:
# dbutils.library.restartPython()

In [0]:
# from databricks_langchain import ChatDatabricks

# chat_model = ChatDatabricks(
#     endpoint="databricks-meta-llama-3-3-70b-instruct",
#     temperature=0.1,
#     max_tokens=250,
# )

# chat_model.invoke("Who is data fudiciary?")

In [0]:
%pip install -U --quiet databricks-sdk "databricks-langchain>=0.4.0" databricks-agents "mlflow[databricks]>=3.10.1" databricks-vectorsearch langchain==0.3.25 langchain-core>=0.3.59 bs4==0.0.2 markdownify==0.14.1 pydantic==2.10.1 openai PyMuPDF

dbutils.library.restartPython()

from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)

chat_model.invoke("Who is data fiduciary?")


In [0]:
from databricks_langchain import DatabricksVectorSearch

vector_store = DatabricksVectorSearch(index_name="workspace.default.documents_index")
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
relevant_documents=retriever.invoke("What is data fiduciary?")

In [0]:
relevant_documents

In [0]:
# For this first basic demo, we'll keep the configuration as a minimum. 
# In real app, you can make all your RAG as a param (such as your prompt template to easily test different prompts!)

chain_config = {
    "llm_model_serving_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",  # the foundation model we want to use
    "vector_search_endpoint_name": "vector_search",  # the endpoint we want to use for vector search
    "vector_search_index": "workspace.default.documents_index",
    "llm_prompt_template": """You are an assistant that answers questions. Use the following pieces of retrieved context to answer the question. Some pieces of context may be irrelevant, in which case you should not use them to form the answer.\n\nContext:\n{context}"""
}

In [0]:
# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

In [0]:
format_context(relevant_documents)

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages(
    [ 
        ("system", chain_config.get("llm_prompt_template")), # Contains the instructions from the configuration
        ("user", "{question}") #user's questions
    ]
)
prompt



In [0]:
from langchain_core.output_parsers import StrOutputParser
from databricks_langchain import ChatDatabricks

# Our foundation model answering the final prompt
chat_model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)
model = chat_model
#Let's try our prompt:
answer = (prompt | model | StrOutputParser()).invoke({'question':'Who is data fudiciary?', 'context': relevant_documents})
answer